# EPIC Clarity — Condition Occurrence Hydration

Populates `_exponent.omop.condition_occurrence` from Epic Clarity diagnosis data.

## Source Tables
- `_exponent._bronze_epic_clarity.pat_enc_dx` — Encounter-level diagnoses
- `_exponent._bronze_epic_clarity.problem_list` — Problem list (longitudinal diagnoses)

## Pipeline
1. `standard_concept_mapping` — ICD9CM / ICD10CM / SNOMED → OMOP standard concept
2. `silver_condition_occurrence` — staged temp view, full OMOP field set
3. MERGE → `omop_silver.condition_occurrence`
4. INSERT → `omop_mapping.source_to_condition_occurrence`
5. `gold` — resolves surrogate IDs and FK references
6. MERGE → `omop.condition_occurrence`

## Dependencies
- `omop_mapping.source_to_person` must be populated for Epic Clarity (person notebook must run first)
- `omop_mapping.source_to_visit_occurrence` must be populated for Epic Clarity

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW standard_concept_mapping AS
WITH ranked AS (
  SELECT
    concept.vocabulary_id         AS source_vocabulary_id,
    concept.concept_code          AS source_concept_code,
    concept.concept_id            AS source_concept_id,
    standard_concept.concept_id   AS standard_concept_id,
    standard_concept.concept_name AS standard_concept_name,
    concept_relationship.valid_start_date AS rel_valid_start_date,
    ROW_NUMBER() OVER (
      PARTITION BY concept.vocabulary_id, concept.concept_code
      ORDER BY concept_relationship.valid_start_date DESC, standard_concept.concept_id ASC
    ) AS rn
  FROM _exponent.omop.concept
  JOIN _exponent.omop.concept_relationship
    ON concept_relationship.concept_id_1 = concept.concept_id
   AND concept_relationship.relationship_id = 'Maps to'
   AND concept_relationship.invalid_reason IS NULL
  JOIN _exponent.omop.concept AS standard_concept
    ON standard_concept.concept_id = concept_relationship.concept_id_2
   AND standard_concept.standard_concept = 'S'
   AND standard_concept.domain_id         = 'Condition'
   AND standard_concept.invalid_reason    IS NULL
  WHERE concept.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')
    AND concept.invalid_reason IS NULL
)
SELECT
  source_vocabulary_id,
  source_concept_code,
  source_concept_id,
  standard_concept_id,
  standard_concept_name
FROM ranked
WHERE rn = 1;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_condition_occurrence AS

-- -------------------------------------------------------
-- Encounter-level diagnoses (PAT_ENC_DX)
-- Unique key: one row per encounter + diagnosis position
-- -------------------------------------------------------
SELECT
  CONCAT_WS(
    CHR(31),
    'epic_clarity',
    'pat_enc_dx',
    'pat_enc_csn_id',
    CAST(ped.PAT_ENC_CSN_ID AS BIGINT)
  )                                                                     AS condition_occurrence_source_value,

  stp.person_id                                                         AS person_id,

  COALESCE(scm.standard_concept_id, 0)                                  AS condition_concept_id,

  CAST(ped.CONTACT_DATE AS DATE)                                        AS condition_start_date,
  CAST(ped.CONTACT_DATE AS TIMESTAMP)                                   AS condition_start_datetime,
  NULL                                                                  AS condition_end_date,
  NULL                                                                  AS condition_end_datetime,

  32020                                                                 AS condition_type_concept_id, -- EHR encounter diagnosis

  NULL                                                                  AS condition_status_concept_id,
  NULL                                                                  AS stop_reason,
  NULL                                                                  AS provider_id,
  NULL                                                                  AS visit_occurrence_id,
  NULL                                                                  AS visit_detail_id,

  NULLIF(TRIM(REGEXP_REPLACE(ped.DX_ID, '[\\s\\u00A0]+', '')), '') AS condition_source_value,

  COALESCE(scm.source_concept_id, 0)                                   AS condition_source_concept_id,
  NULL                                                                  AS condition_status_source_value,

  CONCAT_WS(
    CHR(31),
    'epic_clarity',
    'pat_enc',
    'pat_enc_csn_id',
    CAST(ped.PAT_ENC_CSN_ID AS BIGINT)
  )                                                                     AS visit_occurrence_source_value,

  'epic_clarity'                                                        AS source_system

FROM _exponent._bronze_epic_clarity.pat_enc_dx  ped

INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(
       CHR(31),
       'epic_clarity',
       'PATIENT',
       'PAT_ID',
       ped.PAT_ID
     )
 AND stp.active_flag = TRUE

LEFT JOIN standard_concept_mapping scm
  ON scm.source_concept_code   = NULLIF(TRIM(REGEXP_REPLACE(ped.DX_ID, '[\\s\\u00A0]+', '')), '')
 AND scm.source_vocabulary_id  IN ('ICD9CM', 'ICD10CM', 'SNOMED')

WHERE ped.PAT_ENC_CSN_ID IS NOT NULL
  AND ped.PAT_ID          IS NOT NULL
  AND ped.CONTACT_DATE    IS NOT NULL

UNION ALL

-- -------------------------------------------------------
-- Problem list (PROBLEM_LIST)
-- Unique key: one row per problem list entry (PROBLEM_LIST_ID)
-- -------------------------------------------------------
SELECT
  CONCAT_WS(
    CHR(31),
    'epic_clarity',
    'problem_list',
    'problem_list_id',
    CAST(pl.PROBLEM_LIST_ID AS BIGINT)
  )                                                                     AS condition_occurrence_source_value,

  stp.person_id                                                         AS person_id,

  COALESCE(scm.standard_concept_id, 0)                                  AS condition_concept_id,

  CAST(pl.NOTED_DATE AS DATE)                                           AS condition_start_date,
  CAST(pl.NOTED_DATE AS TIMESTAMP)                                      AS condition_start_datetime,
  CAST(pl.RESOLVED_DATE AS DATE)                                        AS condition_end_date,
  CAST(pl.RESOLVED_DATE AS TIMESTAMP)                                   AS condition_end_datetime,

  32000                                                                 AS condition_type_concept_id, -- Problem list entry

  NULL                                                                  AS condition_status_concept_id,
  NULL                                                                  AS stop_reason,
  NULL                                                                  AS provider_id,
  NULL                                                                  AS visit_occurrence_id,
  NULL                                                                  AS visit_detail_id,

  NULLIF(TRIM(REGEXP_REPLACE(pl.DX_ID, '[\\s\\u00A0]+', '')), '') AS condition_source_value,

  COALESCE(scm.source_concept_id, 0)                                   AS condition_source_concept_id,
  NULL                                                                  AS condition_status_source_value,

  NULL                                                                  AS visit_occurrence_source_value,

  'epic_clarity'                                                        AS source_system

FROM _exponent._bronze_epic_clarity.problem_list pl

INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(
       CHR(31),
       'epic_clarity',
       'PATIENT',
       'PAT_ID',
       pl.PAT_ID
     )
 AND stp.active_flag = TRUE

LEFT JOIN standard_concept_mapping scm
  ON scm.source_concept_code   = NULLIF(TRIM(REGEXP_REPLACE(pl.DX_ID, '[\\s\\u00A0]+', '')), '')
 AND scm.source_vocabulary_id  IN ('ICD9CM', 'ICD10CM', 'SNOMED')

WHERE pl.PROBLEM_LIST_ID IS NOT NULL
  AND pl.PAT_ID           IS NOT NULL
  AND pl.NOTED_DATE       IS NOT NULL;

In [0]:
%sql
MERGE INTO _exponent.omop_silver.condition_occurrence AS target
USING (
  -- Deduplicate on the staging key before MERGE to prevent duplicate row errors
  SELECT *
  FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY condition_occurrence_source_value
        ORDER BY condition_start_date DESC
      ) AS rn
    FROM silver_condition_occurrence
  )
  WHERE rn = 1
) AS source
ON target.condition_occurrence_source_value = source.condition_occurrence_source_value

WHEN MATCHED AND NOT (
     target.person_id                     <=> source.person_id
 AND target.condition_concept_id          <=> source.condition_concept_id
 AND target.condition_start_date          <=> source.condition_start_date
 AND target.condition_start_datetime      <=> source.condition_start_datetime
 AND target.condition_end_date            <=> source.condition_end_date
 AND target.condition_end_datetime        <=> source.condition_end_datetime
 AND target.condition_type_concept_id     <=> source.condition_type_concept_id
 AND target.condition_status_concept_id   <=> source.condition_status_concept_id
 AND target.stop_reason                   <=> source.stop_reason
 AND target.provider_id                   <=> source.provider_id
 AND target.visit_occurrence_id           <=> source.visit_occurrence_id
 AND target.visit_detail_id               <=> source.visit_detail_id
 AND target.condition_source_value        <=> source.condition_source_value
 AND target.condition_source_concept_id   <=> source.condition_source_concept_id
 AND target.condition_status_source_value <=> source.condition_status_source_value
 AND target.source_system                 <=> source.source_system
) THEN UPDATE SET
  target.person_id                     = source.person_id,
  target.condition_concept_id          = source.condition_concept_id,
  target.condition_start_date          = source.condition_start_date,
  target.condition_start_datetime      = source.condition_start_datetime,
  target.condition_end_date            = source.condition_end_date,
  target.condition_end_datetime        = source.condition_end_datetime,
  target.condition_type_concept_id     = source.condition_type_concept_id,
  target.condition_status_concept_id   = source.condition_status_concept_id,
  target.stop_reason                   = source.stop_reason,
  target.provider_id                   = source.provider_id,
  target.visit_occurrence_id           = source.visit_occurrence_id,
  target.visit_detail_id               = source.visit_detail_id,
  target.condition_source_value        = source.condition_source_value,
  target.condition_source_concept_id   = source.condition_source_concept_id,
  target.condition_status_source_value = source.condition_status_source_value,
  target.source_system                 = source.source_system,
  target.last_mod_tsp                  = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_source_value,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value,
  visit_occurrence_source_value,
  source_system,
  last_mod_tsp
) VALUES (
  source.condition_occurrence_source_value,
  source.person_id,
  source.condition_concept_id,
  source.condition_start_date,
  source.condition_start_datetime,
  source.condition_end_date,
  source.condition_end_datetime,
  source.condition_type_concept_id,
  source.condition_status_concept_id,
  source.stop_reason,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.condition_source_value,
  source.condition_source_concept_id,
  source.condition_status_source_value,
  source.visit_occurrence_source_value,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_condition_occurrence (
    source_system,
    condition_occurrence_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    silver_condition.source_system,
    silver_condition.condition_occurrence_source_value,
    TRUE                     AS active_flag,
    CURRENT_TIMESTAMP()      AS created_tsp,
    CURRENT_TIMESTAMP()      AS last_mod_tsp,
    NULL                     AS merge_id,
    NULL                     AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        condition_occurrence_source_value
    FROM _exponent.omop_silver.condition_occurrence
    WHERE condition_occurrence_source_value IS NOT NULL
      AND source_system = 'epic_clarity'
) AS silver_condition
LEFT ANTI JOIN _exponent.omop_mapping.source_to_condition_occurrence AS existing
  ON silver_condition.condition_occurrence_source_value = existing.condition_occurrence_source_value
 AND existing.source_system = 'epic_clarity';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold AS
SELECT
  stco.condition_occurrence_id,
  co.person_id,
  co.condition_concept_id,
  co.condition_start_date,
  co.condition_start_datetime,
  co.condition_end_date,
  co.condition_end_datetime,
  co.condition_type_concept_id,
  co.condition_status_concept_id,
  co.stop_reason,
  co.provider_id,
  COALESCE(stvo.visit_occurrence_id, co.visit_occurrence_id)  AS visit_occurrence_id,
  co.visit_detail_id,
  co.condition_source_value,
  co.condition_source_concept_id,
  co.condition_status_source_value

FROM _exponent.omop_silver.condition_occurrence co

JOIN _exponent.omop_mapping.source_to_condition_occurrence stco
  ON co.condition_occurrence_source_value = stco.condition_occurrence_source_value
 AND stco.source_system = 'epic_clarity'
 AND stco.active_flag   = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
  ON co.visit_occurrence_source_value = stvo.visit_occurrence_source_value
 AND stvo.source_system = 'epic_clarity'
 AND stvo.active_flag   = TRUE

WHERE co.source_system = 'epic_clarity';

In [0]:
%sql
MERGE INTO _exponent.omop.condition_occurrence AS target
USING gold AS source
ON target.condition_occurrence_id = source.condition_occurrence_id

WHEN MATCHED AND NOT (
     target.person_id                     <=> source.person_id
 AND target.condition_concept_id          <=> source.condition_concept_id
 AND target.condition_start_date          <=> source.condition_start_date
 AND target.condition_start_datetime      <=> source.condition_start_datetime
 AND target.condition_end_date            <=> source.condition_end_date
 AND target.condition_end_datetime        <=> source.condition_end_datetime
 AND target.condition_type_concept_id     <=> source.condition_type_concept_id
 AND target.condition_status_concept_id   <=> source.condition_status_concept_id
 AND target.stop_reason                   <=> source.stop_reason
 AND target.provider_id                   <=> source.provider_id
 AND target.visit_occurrence_id           <=> source.visit_occurrence_id
 AND target.visit_detail_id               <=> source.visit_detail_id
 AND target.condition_source_value        <=> source.condition_source_value
 AND target.condition_source_concept_id   <=> source.condition_source_concept_id
 AND target.condition_status_source_value <=> source.condition_status_source_value
) THEN UPDATE SET
  target.person_id                     = source.person_id,
  target.condition_concept_id          = source.condition_concept_id,
  target.condition_start_date          = source.condition_start_date,
  target.condition_start_datetime      = source.condition_start_datetime,
  target.condition_end_date            = source.condition_end_date,
  target.condition_end_datetime        = source.condition_end_datetime,
  target.condition_type_concept_id     = source.condition_type_concept_id,
  target.condition_status_concept_id   = source.condition_status_concept_id,
  target.stop_reason                   = source.stop_reason,
  target.provider_id                   = source.provider_id,
  target.visit_occurrence_id           = source.visit_occurrence_id,
  target.visit_detail_id               = source.visit_detail_id,
  target.condition_source_value        = source.condition_source_value,
  target.condition_source_concept_id   = source.condition_source_concept_id,
  target.condition_status_source_value = source.condition_status_source_value

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_id,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value
) VALUES (
  source.condition_occurrence_id,
  source.person_id,
  source.condition_concept_id,
  source.condition_start_date,
  source.condition_start_datetime,
  source.condition_end_date,
  source.condition_end_datetime,
  source.condition_type_concept_id,
  source.condition_status_concept_id,
  source.stop_reason,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.condition_source_value,
  source.condition_source_concept_id,
  source.condition_status_source_value
);